# 🧪 W7-D1 概念实验：数字员工的行为由什么决定？

> 配套阅读：`第7周-Day1-数字员工总览与Agent行为设计.md`（概念讲解、案例与术语表在那边）
> 本 notebook 只回答一个问题：**「数字员工」不是一个 prompt 字符串，而是 角色 + 权限 + 工具 + 护栏 + 输出契约 的组合体** —— 用 4 个小实验验证每一层配置如何真实地改变行为。
>
> 实验环境：纯 Python 标准库 + numpy/matplotlib 模拟，不调用任何 LLM。


## 实验 1：同一个请求引擎，两份「员工配置」，行为差异从哪来？

数字员工的本质是一份**可执行的配置**：角色只是岗位描述，真正决定行为的是权限集合、可用工具和风险策略。
构建 `DigitalEmployee` dataclass，让两个员工（低权限客服 / 高权限运维）面对同一批请求，观察决策分叉。


In [ ]:
import json
from dataclasses import dataclass, field

@dataclass
class DigitalEmployee:
    name: str
    role: str
    tools: list                    # 可用工具
    permissions: set               # 权限集合
    risk_policy: dict              # {动作: 风险等级}，>=2 需确认，>=3 直接拒绝
    sla_turnaround_s: int = 30

    def dispatch(self, action: str) -> str:
        """请求路由器：权限 → 风险 → 决策"""
        if action not in self.permissions:
            return "拒绝（无权限）"
        risk = self.risk_policy.get(action, 1)
        if risk >= 3:
            return "拒绝（合规红线）"
        if risk == 2:
            return "需人工确认后执行"
        return "直接执行"

requests = ["查询订单", "修改收货地址", "发起退款", "删除数据库", "发送通知"]

support = DigitalEmployee(
    name="客服-小七", role="电商售前售后咨询",
    tools=["订单查询", "政策问答"], permissions={"查询订单", "修改收货地址", "发起退款", "发送通知"},
    risk_policy={"修改收货地址": 2, "发起退款": 2, "发送通知": 1})

ops = DigitalEmployee(
    name="运维-老K", role="系统运维值守",
    tools=["订单查询", "政策问答", "数据库管理"], permissions={"查询订单", "删除数据库", "发送通知"},
    risk_policy={"删除数据库": 3, "发送通知": 1})

print(f"{'请求':<8}| {support.name:<10} | {ops.name:<10}")
print("-" * 44)
for r in requests:
    print(f"{r:<8}| {support.dispatch(r):<12} | {ops.dispatch(r):<12}")

# 观察：同一个动作（发送通知）两人都放行；但「发起退款」客服需确认、运维无权限；
# 「删除数据库」运维有权限却撞上风险=3 的红线 → 权限和护栏是两道独立的闸门

## 实验 2：输出格式 = 交付标准 —— 无契约 vs JSON 契约的机器可解析率

企业里员工交付有模板，数字员工的交付标准就是**输出契约**。
模拟 50 条自由文本回答 vs 50 条走统一 JSON schema 的回答，统计下游程序（`json.loads` + 字段校验）的解析成功率。


In [ ]:
import random

random.seed(7)
REQS = [f"请求#{i}" for i in range(50)]
FIELDS = ["answer", "confidence", "needs_confirmation", "next_step"]

def free_text_answer(q):
    return f"关于{q}：这个问题一般来说是可以解决的，不过具体要看情况……（省略200字）"

def contracted_answer(q, risk):
    return json.dumps({
        "answer": f"{q} 的结论：已按规则处理",
        "confidence": random.choice(["high", "medium", "low"]),
        "needs_confirmation": risk == 2,
        "next_step": "等待用户确认" if risk == 2 else "已办结",
    }, ensure_ascii=False)

def parseable(text):
    try:
        obj = json.loads(text)
        return isinstance(obj, dict) and all(k in obj for k in FIELDS)
    except Exception:
        return False

free_ok = sum(parseable(free_text_answer(q)) for q in REQS)
contract_ok = sum(parseable(contracted_answer(q, random.choice([1, 2]))) for q in REQS)

print(f"自由文本   ：{free_ok}/50 可解析  ({free_ok*2}%)")
print(f"JSON 契约  ：{contract_ok}/50 可解析  ({contract_ok*2}%)")
print("\n结论：输出契约把「能不能接进自动化流水线」从运气变成了工程保证。")

## 实验 3：多轮对话的上下文裁剪 —— 固定 token 预算下，哪种策略关键信息保留率最高？

上下文窗口有限、历史越长噪音越多。常见裁剪策略：① 硬截断（只留最近）② 系统规则+滑窗 ③ 混合（系统永不丢 + 关键消息优先 + 尾部滑窗）。
构造一段 40 轮对话，埋入 5 个关键事实（订单号、退货诉求等），看预算从 200→2000 token 时各策略的保真度。


In [ ]:
import random
from matplotlib import font_manager
import matplotlib.pyplot as plt

font_path = "/usr/share/fonts/opentype/noto/NotoSansCJK-Regular.ttc"
font_manager.fontManager.addfont(font_path)
font_name = font_manager.FontProperties(fname=font_path).get_name()
plt.rcParams["font.family"] = font_name
plt.rcParams["axes.unicode_minus"] = False

random.seed(1)

KEY_FACTS = ["订单号 A1024", "要求退货", "收货地址在上海", "会员等级 VIP", "上次投诉未解决"]

def build_dialog(n=40):
    """构造对话：随机闲聊 + 5 条关键事实散布其中"""
    dialog = [("system", "你是电商客服数字员工，遵守退款与隐私规则。", 28)]
    for i in range(n):
        if i in (5, 12, 20, 28, 35):
            dialog.append(("user", f"关键信息：{KEY_FACTS[(i//7) % 5]}", 12))
        else:
            dialog.append(("user", "闲聊：" + "嗯" * random.randint(5, 20), random.randint(10, 40)))
    return dialog

dialog = build_dialog()

def kept_tokens(strategy, budget):
    if strategy == "hard":
        keep = []
        used = 0
        for idx in range(len(dialog) - 1, -1, -1):
            if used + dialog[idx][2] <= budget:
                keep.append(idx); used += dialog[idx][2]
    elif strategy == "system_window":    # system 永在 + 尾部滑窗
        keep = [0]; used = dialog[0][2]
        for idx in range(len(dialog) - 1, 0, -1):
            if used + dialog[idx][2] <= budget:
                keep.append(idx); used += dialog[idx][2]
    else:                                 # hybrid: system + 关键消息优先 + 尾部
        keep = [0]; used = dialog[0][2]
        for idx, (role, text, tok) in enumerate(dialog):
            if idx and any(k in text for k in ("订单号", "退货", "地址", "VIP", "投诉")):
                if idx not in keep and used + tok <= budget:
                    keep.append(idx); used += tok
        for idx in range(len(dialog) - 1, 0, -1):
            if idx not in keep and used + dialog[idx][2] <= budget:
                keep.append(idx); used += dialog[idx][2]
    return set(keep)

def retention(strategy, budget):
    keep = kept_tokens(strategy, budget)
    hit = sum(1 for idx in keep if "关键信息" in dialog[idx][1])
    return hit / len(KEY_FACTS)

budgets = list(range(200, 2100, 100))
strategies = {"hard": "硬截断", "system_window": "系统+滑窗", "hybrid": "混合策略"}
plt.figure(figsize=(10, 4.5))
for s, label in strategies.items():
    plt.plot(budgets, [retention(s, b) for b in budgets], marker="o", label=label)
plt.xlabel("token 预算"); plt.ylabel("关键事实保留率")
plt.title("上下文裁剪策略：预算 vs 信息保真度")
plt.axhline(1.0, color="gray", ls="--", lw=0.8)
plt.legend(); plt.grid(alpha=0.3); plt.tight_layout(); plt.show()

print("混合策略在低预算时率先拉满：关键消息优先级 > 消息新旧程度")

## 实验 4：护栏矩阵 —— 风险等级 × 员工权限等级如何映射到最终动作？

护栏不是一句话，是一张**决策矩阵**：风险越高、权限越低，动作越保守。
把矩阵显式化，并用 200 条随机请求统计分流比例（执行/确认/升级人工/拒绝）。


In [ ]:
import random
from collections import Counter
import numpy as np
from matplotlib import font_manager
import matplotlib.pyplot as plt

font_path = "/usr/share/fonts/opentype/noto/NotoSansCJK-Regular.ttc"
font_manager.fontManager.addfont(font_path)
font_name = font_manager.FontProperties(fname=font_path).get_name()
plt.rcParams["font.family"] = font_name
plt.rcParams["axes.unicode_minus"] = False

def guard(risk: int, perm: int) -> int:
    """0=直接执行 1=需确认 2=升级人工 3=拒绝"""
    if risk >= 4: return 3
    if risk == 3: return 2 if perm >= 3 else 3
    if risk == 2: return 0 if perm >= 3 else 1
    return 0

matrix = np.array([[guard(r, p) for p in range(1, 5)] for r in range(1, 5)])
labels = ["执行", "确认", "人工", "拒绝"]
colors = ["#8BC34A", "#FFD54F", "#FFB74D", "#E57373"]

fig, axes = plt.subplots(1, 2, figsize=(11, 4.2))
ax = axes[0]
ax.imshow(matrix, cmap="RdYlGn_r", vmin=0, vmax=3)
for i in range(4):
    for j in range(4):
        ax.text(j, i, labels[matrix[i, j]], ha="center", va="center", fontsize=11)
ax.set_xticks(range(4), [f"权限{p}" for p in range(1, 5)])
ax.set_yticks(range(4), [f"风险{r}" for r in range(1, 5)])
ax.set_title("护栏决策矩阵")

random.seed(42)
reqs = [(random.randint(1, 4), random.randint(1, 4)) for _ in range(200)]
dist = Counter(guard(r, p) for r, p in reqs)
ax2 = axes[1]
ax2.bar([labels[k] for k in range(4)], [dist.get(k, 0) for k in range(4)],
        color=[colors[k] for k in range(4)])
for k in range(4):
    ax2.text(k, dist.get(k, 0), f"{dist.get(k,0)}", ha="center", va="bottom")
ax2.set_title("200 条随机请求的分流结果")
plt.tight_layout(); plt.show()

print("要点：矩阵显式化之后，『哪些动作必须拦』从口头约定变成了可测试的代码。")

## 结论

| 配置层 | 实验验证的行为差异 |
|---|---|
| 角色+权限+工具 | 同一请求在两个员工处得到不同决策（实验1） |
| 输出契约 | JSON 契约解析率 100% vs 自由文本 0%（实验2） |
| 上下文策略 | 混合裁剪在低预算下保真度最高（实验3） |
| 护栏矩阵 | 风险×权限显式映射，可单测（实验4） |

**数字员工 = 把「岗位说明书」编译成可执行的决策逻辑。**
→ 深入阅读：同名 `.md` 的 System Prompt 四层结构、SOUL.md 与 OpenClaw vs Hermes 对比。
